In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:

!pip -q install "vllm==0.14.1" "transformers>=4.45" pandas sentencepiece


In [23]:
import os, gc, json, time, math, random, traceback
from pathlib import Path
import pandas as pd
import torch

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "GPU 런타임으로 변경하세요."
print("GPU:", torch.cuda.get_device_name(0))

BASE_MODELS_DIR = Path("/model")
EVAL_JSONL = Path("/content/drive/MyDrive/eval_samples_official_512.jsonl")
OUT_DIR = Path("/content/drive/MyDrive/eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_CSV = OUT_DIR / "model_eval_checkpoint.csv"
FINAL_CSV = OUT_DIR / "model_eval_final.csv"

MODEL_DIRS = {
    "A_anchor": BASE_MODELS_DIR / "out_q_0225_mix50_50",
    "B_mix70_30": BASE_MODELS_DIR / "out_q_0225_mix70_30",
    "C_mix50_50": BASE_MODELS_DIR / "out_q_0225_mix50_50",
}

NUM_SAMPLES = 512
SEED = 42
MAX_SEQ_LEN_FOR_PPL = 256

TP_SIZE = 1
GPU_MEM_UTIL = 0.85
MAX_GEN_TOKS = 16384
TEMPERATURE = 0.0
TOP_P = 1.0

# 필요 시 실제 공유 산식으로 교체
def submission_score(test_time_sec: float, ppl_proxy: float, metrics: dict) -> float:
    # 예시(교체 가능)
    return max(0.0, 0.7 * (1.0 / max(ppl_proxy, 1e-8)) + 0.3 * max(0.0, 1 - test_time_sec / 600.0))

def load_prompts(path: Path, n=512, seed=42):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                t = obj.get("text", "")
                if t:
                    texts.append(t)
    random.Random(seed).shuffle(texts)
    return texts[:n]

PROMPTS = load_prompts(EVAL_JSONL, NUM_SAMPLES, SEED)
print("Loaded prompts:", len(PROMPTS))

def run_vllm_metrics(model_dir: Path, prompts, gpu_mem_util=0.85):
    llm = LLM(
        model=str(model_dir),
        tensor_parallel_size=TP_SIZE,
        gpu_memory_utilization=gpu_mem_util,
        trust_remote_code=True,
        disable_log_stats=True,
        enforce_eager=True,  # 엔진 초기화 실패 완화
    )
    sp = SamplingParams(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_GEN_TOKS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sp)
    elapsed = time.time() - t0

    out_lens, finish = [], []
    for o in outputs:
        oo = o.outputs[0]
        out_lens.append(len(oo.token_ids))
        finish.append(getattr(oo, "finish_reason", "unknown"))

    out_lens_sorted = sorted(out_lens)
    def pct(arr, p): return arr[int((len(arr)-1)*p)]

    finish_counts = {k: finish.count(k) for k in sorted(set(finish))}
    length_finish_ratio = finish_counts.get("length", 0) / max(len(finish), 1)

    del llm
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "test_time_sec": round(elapsed, 3),
        "out_tok_mean": round(sum(out_lens)/len(out_lens), 2),
        "out_tok_p50": pct(out_lens_sorted, 0.50),
        "out_tok_p90": pct(out_lens_sorted, 0.90),
        "out_tok_p95": pct(out_lens_sorted, 0.95),
        "out_tok_max": max(out_lens),
        "length_finish_ratio": round(length_finish_ratio, 4),
        "finish_reason_counts": finish_counts,
    }

def compute_ppl_proxy(model_dir: Path, texts, max_seq_len=256):
    tok = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(model_dir),
        trust_remote_code=True,
        local_files_only=True,
        torch_dtype=torch.bfloat16,
    ).to("cuda")
    model.eval()

    losses = []
    with torch.no_grad():
        for t in texts:
            enc = tok(t, return_tensors="pt", truncation=True, max_length=max_seq_len)
            ids = enc["input_ids"].to("cuda")
            attn = enc["attention_mask"].to("cuda")
            out = model(input_ids=ids, attention_mask=attn, labels=ids)
            if out.loss is not None:
                losses.append(out.loss.item())

    del model
    gc.collect()
    torch.cuda.empty_cache()

    if not losses:
        return float("inf")
    return float(math.exp(sum(losses) / len(losses)))

ckpt_df = pd.DataFrame()
print("Checkpoint disabled: keep results in-memory only")


GPU: Tesla T4
Loaded prompts: 512
Checkpoint disabled: keep results in-memory only


In [26]:
assert (BASE_MODELS_DIR / "out_q_0225_mix50_50").exists()


AssertionError: 

In [27]:
!ls -al /content/drive/MyDrive
!ls -al /content/drive/MyDrive/model

total 9328
-rw------- 1 root root  706332 Nov  2 16:58  406_5_Stock_lab.pdf
drwx------ 2 root root    4096 Feb  8  2025 'Colab Notebooks'
drwx------ 2 root root    4096 Feb  4 06:31  content
drwx------ 2 root root    4096 Feb 24 16:43  eval
-rw------- 1 root root 2200917 Feb 24 14:38  eval_samples_official_512.jsonl
-rw------- 1 root root   13567 Nov  4 06:18  FILE_STRUCTURE_GUIDE.md
-rw------- 1 root root  270217 Sep 30 01:17  IMG_6470.png
-rw------- 1 root root  213523 Sep 30 01:19  IMG_6471.png
drwx------ 2 root root    4096 Oct 28 05:59  jungle
drwx------ 4 root root    4096 Feb 24 16:46  model
drwx------ 2 root root    4096 Oct 27 06:27  models
drwx------ 2 root root    4096 Oct 27 06:26  quant_analysis
-rw------- 1 root root    4094 Oct 31 05:05  stocklab_industry_balanced_100_nodupes.csv
-rw------- 1 root root  174619 Oct 28 06:17  Untitled0.ipynb
drwx------ 2 root root    4096 Oct 28 07:19 '상장 정보 dataset'
drwx------ 2 root root    4096 Mar 12  2025 '청년 정신건강'


In [28]:
!find /content/drive -maxdepth 6 -type d -name "out_q_0225_mix50_50"


/content/drive/MyDrive/model/out_q_0225_mix50_50
/content/drive/.Encrypted/MyDrive/model/out_q_0225_mix50_50


In [ ]:
from pathlib import Path

BASE_MODELS_DIR = Path("/content/drive/MyDrive/model")
MODEL_DIRS = {
    "A_anchor": Path("/content/drive/MyDrive/model/out_q_0225_mix50_50"),
    "B_mix30_70": Path("/content/drive/MyDrive/model/out_q_0225_mix30_70"),  # 실제 폴더명 기준
    "C_mix50_50": Path("/content/drive/MyDrive/model/out_q_0225_mix50_50"),
    "D_keep_03_26" : Path("/content/drive/MyDrive/model/out_q_keep_0_3_26"),
    "E_keep_03_26_sta" : Path("/content/drive/MyDrive/model/out_q_keep_0_3_26_stratified"),
}

print("BASE_MODELS_DIR =", BASE_MODELS_DIR)
print("A exists:", MODEL_DIRS["A_anchor"].exists())
print("B exists:", MODEL_DIRS["B_mix70_30"].exists())
print("C exists:", MODEL_DIRS["C_mix50_50"].exists())


BASE_MODELS_DIR = /content/drive/MyDrive/model
A exists: True
B exists: True
C exists: True


## 0225_mix50_50

In [30]:
# 실행할 모델 이름을 하나씩 바꿔서 실행
RUN_MODEL = "A_anchor"  # <- ??

existing = set(ckpt_df["model_name"].tolist()) if len(ckpt_df) else set()
if RUN_MODEL in existing:
    print(f"[SKIP] already done: {RUN_MODEL}")
else:
    mdir = MODEL_DIRS[RUN_MODEL]
    assert mdir.exists(), f"missing model: {mdir}"

    row = {"model_name": RUN_MODEL, "model_dir": str(mdir)}
    try:
        # vLLM 엔진 초기화 실패(RuntimeError) 대비: 2회 재시도
        try:
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=GPU_MEM_UTIL)
        except RuntimeError:
            print("[WARN] vLLM init failed once, retry with lower gpu_memory_utilization=0.75")
            gc.collect(); torch.cuda.empty_cache(); time.sleep(3)
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=0.75)

        ppl = compute_ppl_proxy(mdir, PROMPTS, MAX_SEQ_LEN_FOR_PPL)
        score = submission_score(vm["test_time_sec"], ppl, vm)

        row.update({
            "status": "success",
            "submission_formula_score": round(score, 8),
            "test_time_sec": vm["test_time_sec"],
            "ppl_proxy": round(ppl, 8),
            "out_tok_mean": vm["out_tok_mean"],
            "out_tok_p50": vm["out_tok_p50"],
            "out_tok_p90": vm["out_tok_p90"],
            "out_tok_p95": vm["out_tok_p95"],
            "out_tok_max": vm["out_tok_max"],
            "length_finish_ratio": vm["length_finish_ratio"],
            "finish_reason_counts": json.dumps(vm["finish_reason_counts"], ensure_ascii=False),
            "error": "",
        })
        print(f"[OK] {RUN_MODEL} done")

    except Exception as e:
        row.update({
            "status": "failed",
            "submission_formula_score": None,
            "test_time_sec": None,
            "ppl_proxy": None,
            "out_tok_mean": None,
            "out_tok_p50": None,
            "out_tok_p90": None,
            "out_tok_p95": None,
            "out_tok_max": None,
            "length_finish_ratio": None,
            "finish_reason_counts": "",
            "error": f"{type(e).__name__}: {str(e)}",
        })
        print("[ERROR]", RUN_MODEL, row["error"])
        print(traceback.format_exc(limit=2))

    # 즉시 체크포인트 저장
    ckpt_df = pd.concat([ckpt_df, pd.DataFrame([row])], ignore_index=True)
    print("Checkpoint save skipped (CSV disabled)")

display(ckpt_df.sort_values(["status","submission_formula_score"], ascending=[True, False]))


INFO 02-24 16:54:03 [utils.py:263] non-default args: {'trust_remote_code': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/drive/MyDrive/model/out_q_0225_mix50_50'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-24 16:54:25 [model.py:530] Resolved architecture: Exaone4ForCausalLM
WARNING 02-24 16:54:25 [model.py:1817] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 02-24 16:54:25 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 02-24 16:54:25 [model.py:1545] Using max model len 65536
INFO 02-24 16:54:25 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-24 16:54:25 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 02-24 16:54:25 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 02-24 16:54:25 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 02-24 16:54:25 [vllm.py:765] Cudagraph is disabled under eager mode


The tokenizer you are loading from '/content/drive/MyDrive/model/out_q_0225_mix50_50' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


WARNING 02-24 16:54:28 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 02-24 16:58:02 [llm.py:347] Supported tasks: ['generate']


Adding requests:   0%|          | 0/512 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/512 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

The tokenizer you are loading from '/content/drive/MyDrive/model/out_q_0225_mix50_50' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
Compressing model: 154it [00:00, 922.34it/s] 


[OK] A_anchor done
Checkpoint save skipped (CSV disabled)


,model_name,model_dir,status,submission_formula_score,test_time_sec,ppl_proxy,out_tok_mean,out_tok_p50,out_tok_p90,out_tok_p95,out_tok_max,length_finish_ratio,finish_reason_counts,error
0,A_anchor,/content/drive/MyDrive/model/out_q_0225_mix50_50,success,0.18919,357.796,10.280813,584.83,622,964,1045,2012,0.0,"{""stop"": 512}",


## out_q_0225_mix30_70


In [ ]:
# 실행할 모델 이름을 하나씩 바꿔서 실행
RUN_MODEL = "B_mix70_30"  # <- ??

existing = set(ckpt_df["model_name"].tolist()) if len(ckpt_df) else set()
if RUN_MODEL in existing:
    print(f"[SKIP] already done: {RUN_MODEL}")
else:
    mdir = MODEL_DIRS[RUN_MODEL]
    assert mdir.exists(), f"missing model: {mdir}"

    row = {"model_name": RUN_MODEL, "model_dir": str(mdir)}
    try:
        # vLLM 엔진 초기화 실패(RuntimeError) 대비: 2회 재시도
        try:
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=GPU_MEM_UTIL)
        except RuntimeError:
            print("[WARN] vLLM init failed once, retry with lower gpu_memory_utilization=0.75")
            gc.collect(); torch.cuda.empty_cache(); time.sleep(3)
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=0.75)

        ppl = compute_ppl_proxy(mdir, PROMPTS, MAX_SEQ_LEN_FOR_PPL)
        score = submission_score(vm["test_time_sec"], ppl, vm)

        row.update({
            "status": "success",
            "submission_formula_score": round(score, 8),
            "test_time_sec": vm["test_time_sec"],
            "ppl_proxy": round(ppl, 8),
            "out_tok_mean": vm["out_tok_mean"],
            "out_tok_p50": vm["out_tok_p50"],
            "out_tok_p90": vm["out_tok_p90"],
            "out_tok_p95": vm["out_tok_p95"],
            "out_tok_max": vm["out_tok_max"],
            "length_finish_ratio": vm["length_finish_ratio"],
            "finish_reason_counts": json.dumps(vm["finish_reason_counts"], ensure_ascii=False),
            "error": "",
        })
        print(f"[OK] {RUN_MODEL} done")

    except Exception as e:
        row.update({
            "status": "failed",
            "submission_formula_score": None,
            "test_time_sec": None,
            "ppl_proxy": None,
            "out_tok_mean": None,
            "out_tok_p50": None,
            "out_tok_p90": None,
            "out_tok_p95": None,
            "out_tok_max": None,
            "length_finish_ratio": None,
            "finish_reason_counts": "",
            "error": f"{type(e).__name__}: {str(e)}",
        })
        print("[ERROR]", RUN_MODEL, row["error"])
        print(traceback.format_exc(limit=2))

    # 즉시 체크포인트 저장
    ckpt_df = pd.concat([ckpt_df, pd.DataFrame([row])], ignore_index=True)
    print("Checkpoint save skipped (CSV disabled)")

display(ckpt_df.sort_values(["status","submission_formula_score"], ascending=[True, False]))


INFO 02-24 17:08:02 [utils.py:263] non-default args: {'trust_remote_code': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/content/drive/MyDrive/model/out_q_0225_mix30_70'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-24 17:08:03 [model.py:530] Resolved architecture: Exaone4ForCausalLM
WARNING 02-24 17:08:03 [model.py:1817] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 02-24 17:08:03 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 02-24 17:08:03 [model.py:1545] Using max model len 65536
INFO 02-24 17:08:03 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 02-24 17:08:03 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 02-24 17:08:03 [vllm.py:765] Cudagraph is disabled under eager mode


The tokenizer you are loading from '/content/drive/MyDrive/model/out_q_0225_mix30_70' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 02-24 17:09:03 [llm.py:347] Supported tasks: ['generate']


Adding requests:   0%|          | 0/512 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/512 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

The tokenizer you are loading from '/content/drive/MyDrive/model/out_q_0225_mix30_70' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Compressing model: 154it [00:00, 1021.16it/s]


## out_q_keep_0_3_26


In [ ]:
# 실행할 모델 이름을 하나씩 바꿔서 실행
RUN_MODEL = "D_keep_03_26"  # <- ??

existing = set(ckpt_df["model_name"].tolist()) if len(ckpt_df) else set()
if RUN_MODEL in existing:
    print(f"[SKIP] already done: {RUN_MODEL}")
else:
    mdir = MODEL_DIRS[RUN_MODEL]
    assert mdir.exists(), f"missing model: {mdir}"

    row = {"model_name": RUN_MODEL, "model_dir": str(mdir)}
    try:
        # vLLM 엔진 초기화 실패(RuntimeError) 대비: 2회 재시도
        try:
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=GPU_MEM_UTIL)
        except RuntimeError:
            print("[WARN] vLLM init failed once, retry with lower gpu_memory_utilization=0.75")
            gc.collect(); torch.cuda.empty_cache(); time.sleep(3)
            vm = run_vllm_metrics(mdir, PROMPTS, gpu_mem_util=0.75)

        ppl = compute_ppl_proxy(mdir, PROMPTS, MAX_SEQ_LEN_FOR_PPL)
        score = submission_score(vm["test_time_sec"], ppl, vm)

        row.update({
            "status": "success",
            "submission_formula_score": round(score, 8),
            "test_time_sec": vm["test_time_sec"],
            "ppl_proxy": round(ppl, 8),
            "out_tok_mean": vm["out_tok_mean"],
            "out_tok_p50": vm["out_tok_p50"],
            "out_tok_p90": vm["out_tok_p90"],
            "out_tok_p95": vm["out_tok_p95"],
            "out_tok_max": vm["out_tok_max"],
            "length_finish_ratio": vm["length_finish_ratio"],
            "finish_reason_counts": json.dumps(vm["finish_reason_counts"], ensure_ascii=False),
            "error": "",
        })
        print(f"[OK] {RUN_MODEL} done")

    except Exception as e:
        row.update({
            "status": "failed",
            "submission_formula_score": None,
            "test_time_sec": None,
            "ppl_proxy": None,
            "out_tok_mean": None,
            "out_tok_p50": None,
            "out_tok_p90": None,
            "out_tok_p95": None,
            "out_tok_max": None,
            "length_finish_ratio": None,
            "finish_reason_counts": "",
            "error": f"{type(e).__name__}: {str(e)}",
        })
        print("[ERROR]", RUN_MODEL, row["error"])
        print(traceback.format_exc(limit=2))

    # 즉시 체크포인트 저장
    ckpt_df = pd.concat([ckpt_df, pd.DataFrame([row])], ignore_index=True)
    print("Checkpoint save skipped (CSV disabled)")

display(ckpt_df.sort_values(["status","submission_formula_score"], ascending=[True, False]))
